# Ch 05 · Your First MJWarp Simulation (and How to See It)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/05-first-simulation.ipynb)

Companion notebook for the chapter page `chapters/05-first-simulation.html`. Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom. Every example also runs on a CPU runtime with small budgets (slower, but correct).

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Rendering notebooks only: let glvnd find the NVIDIA EGL driver (from the official
# mujoco_warp tutorial). Must run before mujoco is imported by the setup cell below.
import os
os.environ.setdefault("MUJOCO_GL", "egl")
NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
try:
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version": "1.0.0", '
                    '"ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')
    print("EGL ICD file:", NVIDIA_ICD_CONFIG_PATH, "| MUJOCO_GL =", os.environ["MUJOCO_GL"])
except OSError as e:
    print("could not write the EGL ICD file (fine outside Colab):", e)

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
os.environ.setdefault("MUJOCO_GL", "egl")
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

In [ ]:
# Optional media helpers used under some examples (Rule O: skip if missing).
try:
    import mediapy as media
except ImportError:
    media = None
    print('skipped: mediapy not installed')

## 1-3. The four calls, `put_model` and `Option`
`put_model` copies a compiled `MjModel` to the device as float32 arrays, checks features and adds warp-only options. Continuous options are arrays of shape `(1,)`; discrete ones are Python ints that are baked into the kernels.

In [ ]:
# Example 5.1 - ex1_option_table.py
"""Example 5.1 - read m.opt: MuJoCo fields vs warp-only fields, arrays vs Python ints.

put_model turns mjm.opt into an mjw.Option. Continuous options become wp.arrays with a leading
batch dimension (shape (1,) by default); discrete options stay plain Python ints/bools, which
are baked into the kernels. Fields that do not exist on mujoco.MjOption are warp-only.
Run: python ex1_option_table.py
"""
import dataclasses

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/>
      <geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/>
      <geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)

mj_names = {n for n in dir(mujoco.MjOption) if not n.startswith("_")}
print(f"{'field':<24} {'kind':<20} {'value':<26} note")
for f in dataclasses.fields(m.opt):
    v = getattr(m.opt, f.name)
    if isinstance(v, wp.array):
        kind = f"array{v.shape} {v.dtype.__name__}"
        x = v.numpy()[0]
        val = f"{float(x):.6g}" if np.ndim(x) == 0 else np.array2string(x, precision=4)
    else:
        kind = type(v).__name__
        val = v.name if hasattr(v, "name") else repr(v)  # enums print their name
    note = "" if f.name in mj_names else "warp-only"
    print(f"{f.name:<24} {kind:<20} {val:<26} {note}")

print()
print("timestep:", m.opt.timestep.numpy(), "| integrator:", m.opt.integrator,
      mjw.IntegratorType(m.opt.integrator).name)
print("broadphase:", mjw.BroadphaseType(m.opt.broadphase).name,
      "| warn_overflow:", m.opt.warn_overflow, "=", int(mjw.OverflowType.ALL))
try:
    m.opt.ls_parallel
except AttributeError as e:
    print("AttributeError:", e)

In [ ]:
# Example 5.2 - ex6_tolerance_clamp.py
"""Example 5.2 - put_model clamps the solver tolerance to 1e-6 for float32.

C MuJoCo's default tolerance (1e-8) was chosen for float64. In float32 the solver could never
reach it and would bounce around the optimum, so put_model applies
opt.tolerance = max(tolerance, 1e-6). The host model is not changed.
Run: python ex6_tolerance_clamp.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=8, suppress=False)  # notebook: undo Example 5.1's print settings

XML = """
<mujoco>
  <worldbody>
    <geom type="plane" size="1 1 .1"/>
    <body pos="0 0 .5"><freejoint/><geom type="box" size=".1 .1 .1"/></body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(XML)
print("MJCF default tolerance (host):", mjm.opt.tolerance)
for tol in (1e-10, 1e-8, 1e-6, 1e-3):
    mjm.opt.tolerance = tol
    m = mjw.put_model(mjm)
    print(f"host tolerance {tol:.0e} -> m.opt.tolerance = {m.opt.tolerance.numpy()}")
print("host value after the loop (put_model never writes it):", mjm.opt.tolerance)
print("float32 machine epsilon:", np.finfo(np.float32).eps)

## 4. `make_data` vs `put_data`
`make_data` starts every world at `qpos0` with derived fields blank until `forward`. `put_data` copies an existing `MjData`, and runs kinematics **on your host copy** first.

In [ ]:
# Example 5.3 - ex2_make_data_blank.py
"""Example 5.3 - make_data leaves derived fields blank until forward fills them.

make_data tiles qpos0 into every world and seeds a few poses (body orientations, static geoms),
but body positions such as d.xpos stay zero until you call mjw.forward (or mjw.step).
forward computes everything except integration, so d.time does not move.
Run: python ex2_make_data_blank.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4, nconmax=4, njmax=16)

print("qpos (tiled qpos0), world 0:", d.qpos.numpy()[0])
print("after make_data : xpos[0, 1] =", d.xpos.numpy()[0, 1], "| time =", d.time.numpy())
print("seeded by make_data: xquat[0, 1] =", d.xquat.numpy()[0, 1],
      "| geom_xpos[0, 1] =", d.geom_xpos.numpy()[0, 1])

mjw.forward(m, d)
print("after forward   : xpos[0, 1] =", d.xpos.numpy()[0, 1], "| time =", d.time.numpy())
print("qacc world 0 (gravity only):", d.qacc.numpy()[0])

mjw.step(m, d)
print("after one step  : time =", d.time.numpy())
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

In [ ]:
# Example 5.4 - ex4_put_data_mutates.py
"""Example 5.4 - put_data copies a host MjData to the device, and changes the host copy too.

A fresh mujoco.MjData has qpos set but no kinematics. put_data runs mj_kinematics on it (and,
because its mass matrix is still all zero, mj_comPos, mj_camlight, mj_tendon, mj_crb and
mj_factorM), so mjd.xpos and mjd.M are filled afterwards. Then it tiles the state x nworld.
Run: python ex4_put_data_mutates.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
mjd = mujoco.MjData(mjm)
mjd.qvel[0] = 0.5                       # give the ball a sideways push on the host

print("host before put_data: xpos[1] =", mjd.xpos[1], "| M all zero:", not mjd.M.any())
d = mjw.put_data(mjm, mjd, nworld=3, nconmax=4, njmax=16)
print("host after  put_data: xpos[1] =", mjd.xpos[1], "| M all zero:", not mjd.M.any())

print("device qvel (every world got the push):")
print(d.qvel.numpy())
print("device xpos[:, 1] (copied, already filled):", d.xpos.numpy()[:, 1].tolist())

## 5-7. `step`, `get_data_into`, float32 accuracy, chaos and the clock

In [ ]:
# Example 5.5 - ex8_get_data_into.py
"""Example 5.5 - get_data_into: copy ONE world back into a mujoco.MjData, and what it costs.

get_data_into filters the shared contact pool down to one world, reorders that world's
constraint rows the way C MuJoCo expects (contacts last and contiguous), and copies well over a
hundred device arrays to the host, one .numpy() at a time. Great for rendering or debugging
one world; far too slow for a training loop, where you read whole batched arrays instead.
Run: python ex8_get_data_into.py
"""
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/>
      <geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/>
      <geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 60
NREP = 20

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
for _ in range(NSTEP):
    mjw.step(m, d)
wp.synchronize()

mjd = mujoco.MjData(mjm)                          # fresh host buffer: ncon = 0, nefc = 0
print("fresh MjData : ncon", mjd.ncon, "nefc", mjd.nefc)
mjw.get_data_into(mjd, mjm, d, world_id=3)
print("world 3      : ncon", mjd.ncon, "nefc", mjd.nefc, "time", round(mjd.time, 4))
print("world 3 qpos :", mjd.qpos)
print("pool holds", int(d.nacon.numpy()[0]), "contacts for all", NWORLD, "worlds; worldids:",
      d.contact.worldid.numpy()[: int(d.nacon.numpy()[0])][:12], "...")
print("world 3 contact geom pairs:", [tuple(int(g) for g in c.geom) for c in mjd.contact])

t0 = time.perf_counter()
for _ in range(NREP):
    mjw.get_data_into(mjd, mjm, d, world_id=3)
t_gdi = (time.perf_counter() - t0) / NREP
t0 = time.perf_counter()
for _ in range(NREP):
    q = d.qpos.numpy()                            # every world's qpos in one copy
t_np = (time.perf_counter() - t0) / NREP
print(f"get_data_into (1 world)     : {t_gdi * 1e3:7.2f} ms per call")
print(f"d.qpos.numpy() (all worlds) : {t_np * 1e3:7.3f} ms per call, shape {q.shape}")
print(f"ratio ~ {t_gdi / t_np:.0f}x (CPU dev box; on a GPU each copy also crosses PCIe)")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

In [ ]:
# Example 5.6 - ex3_step_compare.py
"""Example 5.6 - step MJWarp and C MuJoCo side by side and compare one world.

All NWORLD worlds start from the same state, so each one should track C MuJoCo (float64) up to
float32 rounding. We copy world 2 back with get_data_into and compare qpos, ncon and nefc.
Run: python ex3_step_compare.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/>
      <geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/>
      <geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 60

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)

ref = mujoco.MjData(mjm)          # C MuJoCo, float64, one world
for _ in range(NSTEP):
    mjw.step(m, d)                # all NWORLD worlds in one call
    mujoco.mj_step(mjm, ref)

out = mujoco.MjData(mjm)
mjw.get_data_into(out, mjm, d, world_id=2)

print(f"device: {wp.get_device()} | nworld: {d.nworld} | steps: {NSTEP}")
print(f"time  MJWarp {out.time:.6f}  C {ref.time:.6f}")
print("ball z MJWarp:", out.qpos[2].round(6), " C:", ref.qpos[2].round(6))
print("max |qpos diff| world 2 vs C:", f"{np.abs(out.qpos - ref.qpos).max():.2e}")
print("max |qvel diff| world 2 vs C:", f"{np.abs(out.qvel - ref.qvel).max():.2e}")
print(f"ncon  MJWarp {out.ncon}  C {ref.ncon} | nefc  MJWarp {out.nefc}  C {ref.nefc}")
spread = np.ptp(d.qpos.numpy(), axis=0).max()
print("max spread of qpos across worlds:", f"{spread:.1e}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

### Chaos
The next cell runs 2000 steps in both engines (about 40 s on a CPU runtime, mostly kernel compilation). The cell after it plots the divergence inline.

In [ ]:
# Example 5.7 - ex7_chaos.py
"""Example 5.7 - chaos: float32 MJWarp and float64 C MuJoCo agree, then drift apart.

The double pendulum starts from the "horizontal" keyframe and runs NSTEP steps in both engines.
The two trajectories match to ~1e-7 at first; the double pendulum is chaotic, so that tiny
float32-vs-float64 difference grows exponentially until the motions are unrelated. A second
C MuJoCo run started 1e-9 rad away shows the same growth: it is the system, not a bug.
We switch the model to RK4, which keeps the pendulum's energy; with the default Euler
integrator the pendulum slowly loses energy, settles, and the divergence levels off (--euler).
Run: python ex7_chaos.py [--euler] [--dump-json PATH] [--plot PATH]
  --euler           keep the model's Euler integrator.
  --dump-json PATH  writes decimated |dq| curves and both pendulums' angles for the page chart
                    ("dt" there is the time between kept frames, "timestep" the physics step).
  --plot PATH       saves a log-scale plot (needs matplotlib).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--euler", action="store_true")
parser.add_argument("--dump-json", default=None)
parser.add_argument("--plot", default=None)
args, _ = parser.parse_known_args()

NSTEP = 2000          # 20 s of simulated time; one world is enough for this comparison
DECIM = 10            # keep every 10th step for the page

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
if not args.euler:
    mjm.opt.integrator = mujoco.mjtIntegrator.mjINT_RK4   # energy-preserving enough for 20 s
print("integrator:", mujoco.mjtIntegrator(mjm.opt.integrator).name)
key = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_KEY, "horizontal")

ref = mujoco.MjData(mjm)                      # C MuJoCo, float64
mujoco.mj_resetDataKeyframe(mjm, ref, key)
twin = mujoco.MjData(mjm)                     # C MuJoCo, float64, nudged by 1e-9 rad
mujoco.mj_resetDataKeyframe(mjm, twin, key)
twin.qpos[0] += 1e-9

m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
mjw.reset_data_keyframe(m, d, key)            # MJWarp, float32

dq_warp = np.zeros(NSTEP)
dq_twin = np.zeros(NSTEP)
q_warp = np.zeros((NSTEP, 2))
q_ref = np.zeros((NSTEP, 2))
for i in range(NSTEP):
    mjw.step(m, d)
    mujoco.mj_step(mjm, ref)
    mujoco.mj_step(mjm, twin)
    q_warp[i] = d.qpos.numpy()[0]             # host read every step: fine for 1 world here
    q_ref[i] = ref.qpos
    dq_warp[i] = np.abs(q_warp[i] - q_ref[i]).max()
    dq_twin[i] = np.abs(twin.qpos - ref.qpos).max()

dt = mjm.opt.timestep
for thr in (1e-4, 1e-2, 1.0):
    first = lambda a: f"step {int(np.argmax(a > thr)) + 1}" if (a > thr).any() else "never"
    print(f"first |dq| > {thr:<6g}  MJWarp vs C: {first(dq_warp):<10} "
          f"C vs C+1e-9: {first(dq_twin)}")
print("max |dq| in the first 100 steps (MJWarp vs C):", f"{dq_warp[:100].max():.1e}")
print("q after 20 s  MJWarp:", q_warp[-1], " C:", q_ref[-1])

if args.dump_json:
    sl = slice(DECIM - 1, NSTEP, DECIM)
    lg = lambda a: [round(float(np.log10(max(x, 1e-12))), 3) for x in a]
    # "dt" is the time between kept frames (timestep x DECIM), as in the course replay format
    out = {"integrator": mujoco.mjtIntegrator(mjm.opt.integrator).name,
           "timestep": dt, "dt": dt * DECIM, "nstep": NSTEP, "decim": DECIM,
           "log10_dq_warp": lg(dq_warp[sl]), "log10_dq_twin": lg(dq_twin[sl]),
           "q_warp": np.round(q_warp[sl], 3).tolist(), "q_ref": np.round(q_ref[sl], 3).tolist()}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(out["q_warp"]), "frames")

if args.plot:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("skipped: matplotlib not installed")
    else:
        t = np.arange(1, NSTEP + 1) * dt
        plt.semilogy(t, np.maximum(dq_warp, 1e-12), label="MJWarp (float32) vs C")
        plt.semilogy(t, np.maximum(dq_twin, 1e-12), label="C vs C + 1e-9 rad")
        plt.xlabel("time [s]"); plt.ylabel("max |dq| [rad]"); plt.legend()
        plt.savefig(args.plot, dpi=100)
        print("saved", args.plot)

In [ ]:
# Plot the divergence from Example 5.7 (variables dq_warp, dq_twin, dt are still defined).
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('skipped: matplotlib not installed')
else:
    t = np.arange(1, len(dq_warp) + 1) * dt
    plt.semilogy(t, np.maximum(dq_warp, 1e-12), label='MJWarp (float32) vs C')
    plt.semilogy(t, np.maximum(dq_twin, 1e-12), label='C vs C + 1e-9 rad')
    plt.xlabel('time [s]'); plt.ylabel('max |dq| [rad]'); plt.legend(); plt.show()

### The clock is float32 too
`d.time` is a float32 array; `_next_time` adds the timestep once per step, so long runs accumulate rounding error.

In [ ]:
# Example 5.8 - ex5_time_float32.py
"""Example 5.8 - d.time is float32: it accumulates rounding error step by step.

_next_time adds opt.timestep to d.time once per step, in float32 on the device. After 11 steps
of dt = 0.01 the clock reads 0.10999998, not 0.11. The same running sum in NumPy float32 and
float64 shows where the error comes from and how it grows.
Run: python ex5_time_float32.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

XML = """
<mujoco>
  <option timestep="0.01"/>
  <worldbody>
    <body><joint type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04"/></body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(XML)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2)

for _ in range(11):
    mjw.step(m, d)

t32 = d.time.numpy()[0]
t_py = 0.0
for _ in range(11):
    t_py += 0.01                      # Python float = float64
print(f"MJWarp d.time after 11 steps : {str(t32)}  (dtype {d.time.dtype.__name__})")
print(f"same sum in float64           : {t_py!r}")
print(f"exact                         : {11 * 0.01:.2f}")
print(f"float32 nearest to 0.01       : {float(np.float32(0.01)):.12f}")

# how the error grows: running sums in float32 vs float64 (pure NumPy, no simulation)
for n in (11, 1_000, 100_000, 1_000_000):
    s32 = np.cumsum(np.full(n, 0.01, dtype=np.float32), dtype=np.float32)[-1]
    err = abs(float(s32) - n * 0.01)
    print(f"n = {n:>9,} steps: float32 clock = {float(s32):>16.8f}  error = {err:.2e} s")

## 8. Seeing it
On Colab with EGL, Example 5.9 uses `mujoco.Renderer`; elsewhere it falls back to the MJWarp batch renderer.

In [ ]:
# Example 5.9 - ex9_one_frame.py
"""Example 5.9 - render one frame of world 0, with whichever renderer this machine has.

Path 1 (Colab with EGL): get_data_into world 0 -> mujoco.Renderer (OpenGL) -> uint8 image.
Path 2 (fallback, works on Warp CPU): the MJWarp batch ray tracer, create_render_context at
64x64 -> refit_bvh -> render -> get_rgb, which renders every world at once on the device.
The PNG is written with a tiny standard-library encoder, so no imaging library is needed.
Run: python ex9_one_frame.py [--out frame.png] [--dump-png DIR]
  --dump-png DIR  also writes world 0 at t = 0, 0.25, 0.5 s (batch renderer) for the page.
"""
import argparse
import os
import struct
import zlib

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/>
      <geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/>
      <geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""


def save_png(path, img):
    """Write an (H, W, 3) uint8 array as an 8-bit RGB PNG (standard library only)."""
    h, w, _ = img.shape
    raw = b"".join(b"\x00" + img[y].tobytes() for y in range(h))   # filter type 0 per row

    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data))

    with open(path, "wb") as f:
        f.write(b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


parser = argparse.ArgumentParser()
parser.add_argument("--out", default="frame.png")
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

NWORLD = 2
NSTEP = 50                                   # 0.5 s at dt = 0.01
mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
# chapter-local camera tweak (the shared MJCF stays as it is): move "overview" closer and tilt it
# 30 degrees down (quat = 60 degrees about x), so the objects fill a 64x64 frame
cam = mjm.camera("overview")
cam.pos[:] = (0.25, -1.6, 1.6)
cam.quat[:] = (np.cos(np.pi / 6), np.sin(np.pi / 6), 0.0, 0.0)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)

# the batch renderer: one context for all worlds, built once (it compiles a megakernel)
rc = mjw.create_render_context(mjm, nworld=NWORLD, cam_res=(64, 64), render_rgb=True,
                               render_depth=False, render_seg=False)
rgb = wp.zeros((NWORLD, 64, 64), dtype=wp.vec3)      # (nworld, height, width)


def batch_frame():
    mjw.refit_bvh(m, d, rc)                  # bounding boxes follow the new poses
    mjw.render(m, d, rc)                     # one launch: every world, every camera
    mjw.get_rgb(rc, 0, rgb)                  # unpack camera 0 -> (nworld, H, W) vec3 in [0, 1]
    return (np.clip(rgb.numpy(), 0, 1) * 255).astype(np.uint8)


mjw.forward(m, d)                            # poses at t = 0 (make_data leaves xpos blank)
strip = [batch_frame()[0]]
for i in range(NSTEP):
    mjw.step(m, d)
    if i + 1 in (25, 50):
        strip.append(batch_frame()[0])

try:                                         # path 1: OpenGL renderer (needs EGL/GLFW)
    renderer = mujoco.Renderer(mjm, height=240, width=320)
except Exception as e:                       # ImportError without PyOpenGL/glfw, or no EGL
    print(f"mujoco.Renderer unavailable ({type(e).__name__}); using the MJWarp batch renderer")
    img, path = strip[-1], "batch renderer, 64x64"
else:
    mjd = mujoco.MjData(mjm)
    mjw.get_data_into(mjd, mjm, d, world_id=0)   # copy world 0 back to the host
    renderer.update_scene(mjd, camera="overview")
    img, path = renderer.render(), "mujoco.Renderer, 320x240"

save_png(args.out, img)
print(f"t = {float(d.time.numpy()[0]):.2f} s | path: {path} | image shape {img.shape} {img.dtype}")
print("saved", args.out, "| batch rgb buffer shape:", rgb.shape,
      "| mean pixel:", img.mean().round(1))
if args.dump_png:
    os.makedirs(args.dump_png, exist_ok=True)
    for t, frame in zip(("0.00", "0.25", "0.50"), strip):
        save_png(os.path.join(args.dump_png, f"ex9_world0_t{t}.png"), frame)
    print("wrote", len(strip), "PNGs to", args.dump_png)

In [ ]:
if media is not None:
    media.show_image(img, width=320)

In [ ]:
# Example 5.10 - ex10_video_tiled.py
"""Example 5.10 - a 3-second video of world 0 and a 3x3 tiled frame of 9 worlds.

Nine worlds start identical except for the ball's initial velocity (qvel[:, 0:3]). Every
1/FPS seconds the batch renderer draws all nine worlds in one call; we keep world 0's image as
a video frame and, at the end, tile the nine images into one 3x3 picture (world k goes to row
k // 3, column k % 3).
Run: python ex10_video_tiled.py [--dump-png DIR]
  --dump-png DIR  writes the 3x3 tile at t = 0.6 s and at t = 3 s for the page.
"""
import argparse
import os
import struct
import zlib

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/>
      <geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/>
      <geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""


def save_png(path, img):
    """Write an (H, W, 3) uint8 array as an 8-bit RGB PNG (standard library only)."""
    h, w, _ = img.shape
    raw = b"".join(b"\x00" + img[y].tobytes() for y in range(h))   # filter type 0 per row

    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data))

    with open(path, "wb") as f:
        f.write(b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


def tile(images, ncol):
    """(n, H, W, 3) -> (rows*H, ncol*W, 3); image k lands in row k // ncol, column k % ncol."""
    n, h, w, c = images.shape
    nrow = -(-n // ncol)
    pad = np.zeros((nrow * ncol - n, h, w, c), images.dtype)
    grid = np.concatenate([images, pad]).reshape(nrow, ncol, h, w, c)
    return grid.transpose(0, 2, 1, 3, 4).reshape(nrow * h, ncol * w, c)


parser = argparse.ArgumentParser()
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

NWORLD, RES, FPS, SECONDS = 9, 64, 30, 3.0
mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
cam = mjm.camera("overview")                 # the same closer camera as Example 5.9
cam.pos[:] = (0.25, -1.6, 1.6)
cam.quat[:] = (np.cos(np.pi / 6), np.sin(np.pi / 6), 0.0, 0.0)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)

qvel = np.zeros((NWORLD, mjm.nv), dtype=np.float32)
vx, vz = np.meshgrid([-0.3, 0.0, 0.3], [3.0, 1.5, 0.0])      # 3 x 3 combinations
qvel[:, 0], qvel[:, 2] = vx.ravel(), vz.ravel()               # ball free joint: dofs 0..2
d.qvel.assign(qvel)                                            # write in place

rc = mjw.create_render_context(mjm, nworld=NWORLD, cam_res=(RES, RES), render_rgb=True,
                               render_depth=False, render_seg=False)
rgb = wp.zeros((NWORLD, RES, RES), dtype=wp.vec3)


def render_all():
    mjw.refit_bvh(m, d, rc)
    mjw.render(m, d, rc)
    mjw.get_rgb(rc, 0, rgb)
    return (np.clip(rgb.numpy(), 0, 1) * 255).astype(np.uint8)    # (nworld, H, W, 3)


dt = float(m.opt.timestep.numpy()[0])
render_every = max(1, int(1.0 / (FPS * dt)))                   # same rule as mjwarp-record
nstep = round(SECONDS / dt)
mjw.forward(m, d)
frames, tiles = [render_all()[0]], {}
for i in range(1, nstep + 1):
    mjw.step(m, d)
    if i % render_every == 0:
        imgs = render_all()
        frames.append(imgs[0])
        if i in (60, nstep):
            tiles[i] = tile(imgs, 3)

video = np.stack(frames)
print(f"{nstep} steps, render every {render_every} steps -> {len(frames)} frames of world 0,"
      f" video array {video.shape}")
print("3x3 tile shape:", tiles[nstep].shape, "| ball x per world at the end:",
      d.qpos.numpy()[:, 0].round(2))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

try:                                         # Rule O: optional video writers
    import mediapy as media
    media.write_video("world0.mp4", video, fps=1.0 / (render_every * dt))
    print("saved world0.mp4 with mediapy")
except Exception as e:                       # ImportError, or no ffmpeg
    try:
        from PIL import Image
        ims = [Image.fromarray(f) for f in video]
        ims[0].save("world0.gif", save_all=True, append_images=ims[1:],
                    duration=int(1000 * render_every * dt), loop=0)
        print(f"saved world0.gif with PIL (mediapy: {type(e).__name__})")
    except ImportError:
        print("skipped video file: neither mediapy nor PIL is installed")

if args.dump_png:
    os.makedirs(args.dump_png, exist_ok=True)
    for i, img in tiles.items():
        save_png(os.path.join(args.dump_png, f"ex10_tile_step{i}.png"), img)
    print("wrote", len(tiles), "tiles to", args.dump_png)

In [ ]:
if media is not None:
    media.show_video(video, fps=1.0 / (render_every * dt), width=256)
    media.show_image(tiles[nstep], width=384)

## 9. The command-line tools
`mjwarp-record` runs only on a GPU in 4.0.1; on CPU the example runs the same loop by hand. `mjwarp-viewer` needs a display, so it is not usable inside Colab.

In [ ]:
# Example 5.11 - ex11_record.py
"""Example 5.11 - record a GIF: mjwarp-record on a GPU, the same loop by hand on CPU.

mjwarp-record is GPU only in 4.0.1: cli.unroll calls wp.get_stream() for every step, which raises
"Device cpu is not a CUDA device" on CPU. So on a GPU we call the CLI (RGB, then depth, with the
batch renderer: --render_mode warp), and on CPU we run the loop record.py runs ourselves:
step -> refit_bvh -> render -> get_rgb, every render_every steps, then save with PIL.
Run: python ex11_record.py [--nstep N] [--dump-png DIR]
  --dump-png DIR  writes 3 of the recorded frames (first, middle, last) for the page.
"""
import argparse
import importlib.util
import os
import shutil
import struct
import subprocess
import zlib

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

def save_png(path, img):
    """Write an (H, W, 3) uint8 array as an 8-bit RGB PNG (standard library only)."""
    h, w, _ = img.shape
    raw = b"".join(b"\x00" + img[y].tobytes() for y in range(h))   # filter type 0 per row

    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data))

    with open(path, "wb") as f:
        f.write(b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


parser = argparse.ArgumentParser()
parser.add_argument("--nstep", type=int, default=100)
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
HUMANOID = str(epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml")
HAS_CLI = shutil.which("mjwarp-record") is not None
HAS_PIL = importlib.util.find_spec("PIL") is not None

if ON_GPU and HAS_CLI and HAS_PIL:
    base = ["mjwarp-record", HUMANOID, "--nworld", "1", "--nstep", str(args.nstep),
            "--render_mode", "warp"]
    subprocess.run(base + ["--output", "humanoid.gif"], check=True)
    subprocess.run(base + ["--channel", "depth", "--output", "depth.gif"], check=True)
    print("wrote humanoid.gif and depth.gif")
else:
    print(f"skipped mjwarp-record (GPU: {ON_GPU}, CLI on PATH: {HAS_CLI}, PIL: {HAS_PIL});"
          " running the same loop by hand")
    mjm = mujoco.MjModel.from_xml_path(HUMANOID)
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=1, nconmax=32, njmax=128)
    rc = mjw.create_render_context(mjm, nworld=1, cam_res=(64, 64), render_rgb=True,
                                   render_depth=False, render_seg=False)
    rgb = wp.zeros((1, 64, 64), dtype=wp.vec3)
    render_every = max(1, int(1.0 / (30 * mjm.opt.timestep)))     # record.py's rule, 30 fps
    frames = []
    for step in range(args.nstep):
        mjw.step(m, d)
        if step % render_every == 0:
            mjw.refit_bvh(m, d, rc)
            mjw.render(m, d, rc)
            mjw.get_rgb(rc, 0, rgb)                                  # camera index 0
            frames.append((np.clip(rgb.numpy()[0], 0, 1) * 255).astype(np.uint8))
    print(f"ncam {mjm.ncam} | dt {mjm.opt.timestep} | render every {render_every} steps ->"
          f" {len(frames)} frames, each {frames[0].shape}")
    bits = int(d.overflow.numpy()[0])
    print("overflow:", [o.name for o in mjw.OverflowType if bits & o] or "none")
    if HAS_PIL:
        from PIL import Image
        ims = [Image.fromarray(f) for f in frames]
        ims[0].save("out.gif", save_all=True, append_images=ims[1:],
                    duration=int(render_every * mjm.opt.timestep * 1000), loop=0)
        print("saved out.gif")
    else:
        print("skipped: PIL not installed (pip install pillow) - no GIF written")
    if args.dump_png:
        os.makedirs(args.dump_png, exist_ok=True)
        for k in (0, len(frames) // 2, len(frames) - 1):
            save_png(os.path.join(args.dump_png, f"ex11_frame{k:02d}.png"), frames[k])
        print("wrote 3 PNGs to", args.dump_png)

In [ ]:
import os
from IPython.display import Image, display
for name in ('humanoid.gif', 'depth.gif', 'out.gif'):
    if os.path.exists(name):
        display(Image(filename=name))

## Exercise 5.1: contacts survive `put_data`
Step `BALL_DROP` in C MuJoCo until the ball rests, copy it to 4 worlds with `put_data`, and assert `d.nacon > 0`.

In [ ]:
# Exercise 5.1: put_data from a settled MjData carries its contacts to the device.
# 1. Step BALL_DROP in C MuJoCo until the ball rests on the floor.
# 2. Copy it to 4 worlds with mjw.put_data (explicit nconmax / njmax).
# 3. Assert that the device contact pool is not empty.
import mujoco
import numpy as np
import warp as wp
import mujoco_warp as mjw

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
mjd = mujoco.MjData(mjm)
# ... your code: settle mjd, then d = mjw.put_data(...)
d = None
assert d is not None and d.nacon.numpy()[0] > 0, "no contacts on the device"
print("Exercise 5.1 passed")

In [ ]:
#@title Exercise 5.1: solution (double-click to show the code)
mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
mjd = mujoco.MjData(mjm)
for _ in range(1000):                       # 2 s: the ball has landed and settled
    mujoco.mj_step(mjm, mjd)
print("host ncon:", mjd.ncon, "nefc:", mjd.nefc)          # 1 contact, 4 rows (pyramidal)
d = mjw.put_data(mjm, mjd, nworld=4, nconmax=4, njmax=16)
print("device nacon:", d.nacon.numpy(), "nefc per world:", d.nefc.numpy())
assert d.nacon.numpy()[0] > 0
print("Exercise 5.1 passed")

## Exercise 5.2: match C MuJoCo
Assert that 100 MJWarp steps of `BALL_DROP` match `mujoco.mj_step` within 1e-4 (max over qpos).

In [ ]:
# Exercise 5.2: 100 MJWarp steps of BALL_DROP match C MuJoCo within 1e-4.
# Use make_data with nworld=2, step both engines 100 times, compare world 1 with C.
mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
# ... your code
err = None
assert err is not None and err < 1e-4, err
print(f"max |qpos diff| = {err:.2e}  Exercise 5.2 passed")

In [ ]:
#@title Exercise 5.2: solution (double-click to show the code)
mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2, nconmax=4, njmax=16)
ref = mujoco.MjData(mjm)
for _ in range(100):
    mjw.step(m, d)
    mujoco.mj_step(mjm, ref)
err = np.abs(d.qpos.numpy()[1] - ref.qpos).max()   # one host read, after the loop
assert err < 1e-4, err
print(f"max |qpos diff| = {err:.2e}  Exercise 5.2 passed")   # about 1e-7 on CPU

## Stretch: find a model MJWarp refuses
Write (or find) an MJCF that compiles in MuJoCo but that `mjw.put_model` rejects. Print the exception and explain, in one sentence, why MJWarp cannot run it. Hints: solver, plugins, dense Jacobians, sleeping.

In [ ]:
# your model here